# IAM roles

Creates four **IAM roles**, one per service, each allowed to do only what its job needs.
Run the cells top to bottom. Each cell is safe to run twice, and prints the AWS CLI command it runs (`$ aws ...`).
The account id is masked in all output.

A role has two policies:
- **Trust policy**: *who* may use the role (e.g. only the Lambda service).
- **Permissions policy**: *what* the role may do (e.g. read `landing/`, write `bronze/`).

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# aws() runs the AWS CLI and prints the command and its output (see infra/aws_helpers.py)
from aws_helpers import aws, aws_json, whoami, account_id, SETTINGS, REGION, LAKE, RESULTS

print("identity:", whoami())
assert REGION == "us-east-1", "this project is built for us-east-1"

identity: user/Ahmed


## 2. Policy templates
The policy files in `infra/policies/` contain placeholders such as `__LAKE__` and `__ACCOUNT__`.
`render_policy` fills them from `config/settings.json` and checks the result is valid JSON.

In [2]:
PLACEHOLDERS = {
    "__LAKE__": LAKE,
    "__RESULTS__": RESULTS,
    "__REGION__": REGION,
    "__ACCOUNT__": account_id(),
    "__LAMBDA__": SETTINGS["lambda_name"],
    "__GLUE_JOB__": SETTINGS["glue_job_name"],
    "__WORKGROUP__": SETTINGS["athena_workgroup"],
    "__STATE_MACHINE__": SETTINGS["state_machine_name"],
    "__DB_SILVER__": SETTINGS["glue_databases"]["silver"],
    "__DB_GOLD__": SETTINGS["glue_databases"]["gold"],
}


def render_policy(filename):
    """Read a policy file, fill in the placeholders and return it as compact JSON text."""
    text = (ROOT / "infra" / "policies" / filename).read_text()
    for placeholder, value in PLACEHOLDERS.items():
        text = text.replace(placeholder, value)
    assert "__" not in text, f"{filename}: unfilled placeholder"
    return json.dumps(json.loads(text))      # parses the JSON, so a typo fails here

## 3. Role function
For one role: **create** it (or update its trust policy if it exists), **tag** it, **attach** any AWS-managed policy,
and **attach** our own permissions policy. `put-role-policy` and `attach-role-policy` can be repeated safely.

In [3]:
def ensure_role(name, description, trust_file, permissions_file, managed_policies=()):
    trust = render_policy(trust_file)
    exists = aws("iam", "get-role", "--role-name", name, check=False, show=False).returncode == 0

    if exists:
        print(f"role {name} already exists - updating its trust policy\n")
        aws("iam", "update-assume-role-policy", "--role-name", name, "--policy-document", trust)
    else:
        aws("iam", "create-role", "--role-name", name, "--description", description,
            "--assume-role-policy-document", trust, "--query", "Role.Arn", "--output", "text")

    aws("iam", "tag-role", "--role-name", name, "--tags", "Key=project,Value=patient-flow")
    aws("iam", "update-role", "--role-name", name, "--description", description)   # keeps the description in sync

    for policy_arn in managed_policies:
        aws("iam", "attach-role-policy", "--role-name", name, "--policy-arn", policy_arn)

    aws("iam", "put-role-policy", "--role-name", name, "--policy-name", f"{name}-permissions",
        "--policy-document", render_policy(permissions_file))

## 4. pf-lambda-role
Used by the ingest Lambda. Reads `landing/`, writes `bronze/`. Writes logs through AWS's basic Lambda policy.

In [4]:
ensure_role(
    "pf-lambda-role",
    "Ingest Lambda: copy landing batches to bronze",
    trust_file="policy_lambda.json",
    permissions_file="permissions_lambda.template.json",
    managed_policies=["arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"],
)

$ aws iam get-role --role-name pf-lambda-role


role pf-lambda-role already exists - updating its trust policy

$ aws iam update-assume-role-policy --role-name pf-lambda-role --policy-document {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}


$ aws iam tag-role --role-name pf-lambda-role --tags Key=project,Value=patient-flow


$ aws iam update-role --role-name pf-lambda-role --description Ingest Lambda: copy landing batches to bronze


$ aws iam attach-role-policy --role-name pf-lambda-role --policy-arn arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole


$ aws iam put-role-policy --role-name pf-lambda-role --policy-name pf-lambda-role-permissions --policy-document {"Version": "2012-10-17", "Statement": [{"Sid": "ListLandingAndBronze", "Effect": "Allow", "Action": "s3:ListBucket", "Resource": "arn:aws:s3:::patientflow-datalake-zhmkbf", "Condition": {"StringLike": {"s3:prefix": ["landing/*", "bronze/*"]}}}, {"Sid": "ReadLanding", "Effect": "Allow", "Action": "s3:GetObject", "Resource": "arn:aws:s3:::patientflow-datalake-zhmkbf/landing/*"}, {"Sid": "WriteBronze", "Effect": "Allow", "Action": "s3:PutObject", "Resource": "arn:aws:s3:::patientflow-datalake-zhmkbf/bronze/*"}]}


## 5. pf-glue-role
Used by the Glue job. Reads `bronze/` and `glue_scripts/`, reads and writes `silver/` and `temp/`.
AWS's `AWSGlueServiceRole` adds the Glue service permissions and CloudWatch logging.

In [5]:
ensure_role(
    "pf-glue-role",
    "Glue job: flatten raw FHIR into silver tables",
    trust_file="policy_glue.json",
    permissions_file="permissions_glue.template.json",
    managed_policies=["arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole"],
)

$ aws iam get-role --role-name pf-glue-role


role pf-glue-role already exists - updating its trust policy

$ aws iam update-assume-role-policy --role-name pf-glue-role --policy-document {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "glue.amazonaws.com"}, "Action": "sts:AssumeRole"}]}


$ aws iam tag-role --role-name pf-glue-role --tags Key=project,Value=patient-flow


$ aws iam update-role --role-name pf-glue-role --description Glue job: flatten raw FHIR into silver tables


$ aws iam attach-role-policy --role-name pf-glue-role --policy-arn arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole


$ aws iam put-role-policy --role-name pf-glue-role --policy-name pf-glue-role-permissions --policy-document {"Version": "2012-10-17", "Statement": [{"Sid": "ListLakeBucket", "Effect": "Allow", "Action": ["s3:ListBucket", "s3:GetBucketLocation"], "Resource": "arn:aws:s3:::patientflow-datalake-zhmkbf"}, {"Sid": "ReadBronzeAndScripts", "Effect": "Allow", "Action": "s3:GetObject", "Resource": ["arn:aws:s3:::patientflow-datalake-zhmkbf/bronze/*", "arn:aws:s3:::patientflow-datalake-zhmkbf/glue_scripts/*"]}, {"Sid": "ReadWriteSilverAndTemp", "Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"], "Resource": ["arn:aws:s3:::patientflow-datalake-zhmkbf/silver/*", "arn:aws:s3:::patientflow-datalake-zhmkbf/temp/*"]}]}


## 6. pf-step-functions-role
Used by Step Functions. Invokes the ingest Lambda, runs the Glue job, runs Athena queries in `pf-wg`,
manages tables in the two Glue databases, uses the lake and results buckets, and writes workflow logs.

In [6]:
ensure_role(
    "pf-step-functions-role",
    "Step Functions: orchestrate the daily pipeline",
    trust_file="policy_step_functions.json",
    permissions_file="permissions_step_functions.template.json",
)

$ aws iam get-role --role-name pf-step-functions-role


role pf-step-functions-role already exists - updating its trust policy

$ aws iam update-assume-role-policy --role-name pf-step-functions-role --policy-document {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "states.amazonaws.com"}, "Action": "sts:AssumeRole"}]}


$ aws iam tag-role --role-name pf-step-functions-role --tags Key=project,Value=patient-flow


$ aws iam update-role --role-name pf-step-functions-role --description Step Functions: orchestrate the daily pipeline


$ aws iam put-role-policy --role-name pf-step-functions-role --policy-name pf-step-functions-role-permissions --policy-document {"Version": "2012-10-17", "Statement": [{"Sid": "InvokeIngestLambda", "Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": ["arn:aws:lambda:us-east-1:<ACCOUNT_ID>:function:pf-ingest", "arn:aws:lambda:us-east-1:<ACCOUNT_ID>:function:pf-ingest:*"]}, {"Sid": "RunGlueJob", "Effect": "Allow", "Action": ["glue:StartJobRun", "glue:GetJobRun", "glue:GetJobRuns", "glue:BatchStopJobRun"], "Resource": "arn:aws:glue:us-east-1:<ACCOUNT_ID>:job/pf-bronze-to-silver"}, {"Sid": "RunAthenaQueries", "Effect": "Allow", "Action": ["athena:StartQueryExecution", "athena:GetQueryExecution", "athena:GetQueryResults", "athena:StopQueryExecution", "athena:GetWorkGroup"], "Resource": "arn:aws:athena:us-east-1:<ACCOUNT_ID>:workgroup/pf-wg"}, {"Sid": "GlueCatalogForAthena", "Effect": "Allow", "Action": ["glue:GetDatabase", "glue:GetDatabases", "glue:GetTable", "glue:GetTables"

## 7. pf-trigger-role
Used by EventBridge. When a new batch arrives, EventBridge uses this role to start the pipeline's state machine. It may do nothing else.

In [7]:
ensure_role(
    "pf-trigger-role",
    "EventBridge: start the pipeline when a batch arrives",
    trust_file="policy_trigger.json",
    permissions_file="permissions_trigger.template.json",
)

$ aws iam get-role --role-name pf-trigger-role


$ aws iam create-role --role-name pf-trigger-role --description EventBridge: start the pipeline when a batch arrives --assume-role-policy-document {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "events.amazonaws.com"}, "Action": "sts:AssumeRole"}]} --query Role.Arn --output text


arn:aws:iam::<ACCOUNT_ID>:role/pf-trigger-role
$ aws iam tag-role --role-name pf-trigger-role --tags Key=project,Value=patient-flow


$ aws iam update-role --role-name pf-trigger-role --description EventBridge: start the pipeline when a batch arrives


$ aws iam put-role-policy --role-name pf-trigger-role --policy-name pf-trigger-role-permissions --policy-document {"Version": "2012-10-17", "Statement": [{"Sid": "StartThePipeline", "Effect": "Allow", "Action": "states:StartExecution", "Resource": "arn:aws:states:us-east-1:<ACCOUNT_ID>:stateMachine:pf-batch-pipeline"}]}


### 7b. Remove the unused scheduler role
`pf-scheduler-role` was created for a daily schedule that the project replaced by the event trigger. A role must have its policies detached and deleted before it can be deleted.

In [8]:
OLD = "pf-scheduler-role"
if aws("iam", "get-role", "--role-name", OLD, check=False, show=False).returncode == 0:
    for policy in aws_json("iam", "list-attached-role-policies", "--role-name", OLD)["AttachedPolicies"]:
        aws("iam", "detach-role-policy", "--role-name", OLD, "--policy-arn", policy["PolicyArn"])
    for name in aws_json("iam", "list-role-policies", "--role-name", OLD)["PolicyNames"]:
        aws("iam", "delete-role-policy", "--role-name", OLD, "--policy-name", name)
    aws("iam", "delete-role", "--role-name", OLD)
else:
    print(f"{OLD} is already gone")

$ aws iam get-role --role-name pf-scheduler-role


$ aws iam delete-role-policy --role-name pf-scheduler-role --policy-name pf-scheduler-role-permissions


$ aws iam delete-role --role-name pf-scheduler-role


## 8. Verify
**Who can use each role** (trust policy) and **what each role can do** (permissions).
IAM's policy simulator answers "would this role be allowed to do X on Y?" without doing it.
We test what each role *must* be able to do, and, just as important, what it *must not* be able to do.

In [9]:
ROLE_SERVICE = {
    "pf-lambda-role": "lambda.amazonaws.com",
    "pf-glue-role": "glue.amazonaws.com",
    "pf-step-functions-role": "states.amazonaws.com",
    "pf-trigger-role": "events.amazonaws.com",
}

role_arns = {}
for name, service in ROLE_SERVICE.items():
    role = aws_json("iam", "get-role", "--role-name", name)["Role"]
    principal = role["AssumeRolePolicyDocument"]["Statement"][0]["Principal"]["Service"]
    assert principal == service, f"{name}: trusts {principal}, expected {service}"
    role_arns[name] = role["Arn"]
    print(f"OK  {name}: trusted by {principal}")

OK  pf-lambda-role: trusted by lambda.amazonaws.com


OK  pf-glue-role: trusted by glue.amazonaws.com


OK  pf-step-functions-role: trusted by states.amazonaws.com


OK  pf-trigger-role: trusted by events.amazonaws.com


In [10]:
S3 = "arn:aws:s3:::"
A, R = account_id(), REGION
JOB = SETTINGS["glue_job_name"]
WG = SETTINGS["athena_workgroup"]
SM = SETTINGS["state_machine_name"]

# (role, action, resource, expected decision)
CHECKS = [
    ("pf-lambda-role", "s3:GetObject", f"{S3}{LAKE}/landing/batch_01/a.json", "allowed"),
    ("pf-lambda-role", "s3:PutObject", f"{S3}{LAKE}/bronze/patient_records/a.json",         "allowed"),
    ("pf-lambda-role", "s3:PutObject", f"{S3}{LAKE}/landing/a.json",          "implicitDeny"),
    ("pf-lambda-role", "s3:PutObject", f"{S3}{LAKE}/silver/a.parquet",        "implicitDeny"),

    ("pf-glue-role", "s3:GetObject",    f"{S3}{LAKE}/bronze/patient_records/a.json",        "allowed"),
    ("pf-glue-role", "s3:PutObject",    f"{S3}{LAKE}/silver/a.parquet",       "allowed"),
    ("pf-glue-role", "s3:DeleteObject", f"{S3}{LAKE}/silver/a.parquet",       "allowed"),
    ("pf-glue-role", "s3:PutObject",    f"{S3}{LAKE}/bronze/patient_records/a.json",        "implicitDeny"),
    ("pf-glue-role", "s3:GetObject",    f"{S3}{LAKE}/gold/a.parquet",         "implicitDeny"),

    ("pf-step-functions-role", "lambda:InvokeFunction",      f"arn:aws:lambda:{R}:{A}:function:{SETTINGS['lambda_name']}", "allowed"),
    ("pf-step-functions-role", "glue:StartJobRun",           f"arn:aws:glue:{R}:{A}:job/{JOB}",                            "allowed"),
    ("pf-step-functions-role", "athena:StartQueryExecution", f"arn:aws:athena:{R}:{A}:workgroup/{WG}",                     "allowed"),
    ("pf-step-functions-role", "s3:PutObject",               f"{S3}{LAKE}/gold/a.parquet",                                 "allowed"),
    ("pf-step-functions-role", "s3:PutObject",               f"{S3}{RESULTS}/query.csv",                                   "allowed"),
    ("pf-step-functions-role", "athena:StartQueryExecution", f"arn:aws:athena:{R}:{A}:workgroup/primary",                  "implicitDeny"),

    ("pf-trigger-role", "states:StartExecution", f"arn:aws:states:{R}:{A}:stateMachine:{SM}",     "allowed"),
    ("pf-trigger-role", "states:StartExecution", f"arn:aws:states:{R}:{A}:stateMachine:other",    "implicitDeny"),
    ("pf-trigger-role", "s3:GetObject",          f"{S3}{LAKE}/landing/a.json",                    "implicitDeny"),
]

failures = []
for role, action, resource, expected in CHECKS:
    result = aws_json("iam", "simulate-principal-policy", "--policy-source-arn", role_arns[role],
                      "--action-names", action, "--resource-arns", resource)["EvaluationResults"][0]
    decision = result["EvalDecision"]
    mark = "OK " if decision == expected else "FAIL"
    shown = resource.replace(LAKE, "<lake>").replace(RESULTS, "<results>").replace(A, "<ACCOUNT_ID>")
    print(f"{mark} {role:<24} {action:<27} {shown:<68} {decision}")
    if decision != expected:
        failures.append((role, action, resource, expected, decision))

assert not failures, f"{len(failures)} permission checks failed"
print(f"\nall {len(CHECKS)} permission checks passed")

OK  pf-lambda-role           s3:GetObject                arn:aws:s3:::<lake>/landing/batch_01/a.json                          allowed


OK  pf-lambda-role           s3:PutObject                arn:aws:s3:::<lake>/bronze/patient_records/a.json                    allowed


OK  pf-lambda-role           s3:PutObject                arn:aws:s3:::<lake>/landing/a.json                                   implicitDeny


OK  pf-lambda-role           s3:PutObject                arn:aws:s3:::<lake>/silver/a.parquet                                 implicitDeny


OK  pf-glue-role             s3:GetObject                arn:aws:s3:::<lake>/bronze/patient_records/a.json                    allowed


...

OK  pf-step-functions-role   s3:PutObject                arn:aws:s3:::<results>/query.csv                                     allowed


OK  pf-step-functions-role   athena:StartQueryExecution  arn:aws:athena:us-east-1:<ACCOUNT_ID>:workgroup/primary              implicitDeny


OK  pf-trigger-role          states:StartExecution       arn:aws:states:us-east-1:<ACCOUNT_ID>:stateMachine:pf-batch-pipeline allowed


OK  pf-trigger-role          states:StartExecution       arn:aws:states:us-east-1:<ACCOUNT_ID>:stateMachine:other             implicitDeny


OK  pf-trigger-role          s3:GetObject                arn:aws:s3:::<lake>/landing/a.json                                   implicitDeny

all 18 permission checks passed


## What these roles are for

| Role | Used by | Phase |
|---|---|---|
| `pf-lambda-role` | the ingest Lambda `pf-ingest` | 4 |
| `pf-glue-role` | the Glue job `pf-bronze-to-silver` | 5 |
| `pf-step-functions-role` | the Step Functions workflow `pf-daily-pipeline` | 7 |
| `pf-trigger-role` | the EventBridge rule `pf-new-batch-rule` (starts the pipeline when a batch arrives) | 10 |

The roles are created before the services that use them exist. A policy can name a resource that does not exist yet.

**Next:** `upload_and_test` uploads the data batches and runs the first Athena query.